# Implementing Canonicalizing Open Knowledge Bases 

### Paper Title: Canonicalizing Open Knowledge Bases
Authors: Luis Galárraga, Geremy Heitz, Kevin Murphy, and Fabian Suchanek (CIKM 2014)

## Motivation and Problem Statement

### The Problem
Open Information Extraction (Open IE) systems like ReVerb or NELL extract facts from web text as triples:
$⟨subject,predicate,object⟩$<br>
For example:
- $⟨ Obama, was born in, Honolulu⟩$
- $⟨ Barack Obama, place of birth, Honolulu⟩$

But as we know:
- Obama and Barack Obama are the same person as well as
- was born in and place of birth are same relation.

These systems as we can see do not canonicalize names or relations which leads to:
- Redundancy
- Ambiguity, and
- Incomplete or polluted knowledge queries.




### Goal:
Canonicalize entities and relations in large Open IE knowledge bases by:
- Clustering synonymous entity mentions (e.g., "Obama", "President Obama")
- Clustering semantically equivalent relations (e.g., "was born in", "birthplace is")

This bridges the gap between:
- Open IE: High recall, low precision
- Closed IE: High precision, low coverage

## Assumptions, Goals, and Research Questions

### Assumptions:
- Mentions in the same web page refer to the same entity.
- Clustering can recover real-world canonical forms from noisy extractions.
- Features like string similarity, attribute overlap, and type constraints help identify synonymy.

### Main Research Questions:
- Can we cluster noun phrases into canonical entity groups?
- Can we cluster relation phrases into semantically coherent groups?
- What features and design choices lead to the most effective clustering?

## Methodology

### **Overview of Pipeline:**

#### A. Entity Canonicalization

* **Input**: Triples from ReVerb or NELL:
  $\langle \text{subject}, \text{relation}, \text{object} \rangle$
* **Step 1**: Create “mentions” (noun phrases with context)
* **Step 2**: Cluster mentions using Hierarchical Agglomerative Clustering (HAC)
* **Step 3**: Use blocking (canopies) to improve HAC efficiency
* **Step 4**: Define various similarity functions between mentions
* **Step 5**: Learn a weighted similarity metric using logistic regression


#### B. Relation Canonicalization
* **Step 1**: Use AMIE (a rule mining system) to find **equivalent relations**
* **Step 2**: Convert subsumption rules into **clusters of equivalent phrases**
* **Step 3**: Optionally map clusters to **Freebase relations**


### A. **Mathematical Models: Entity Similarity and Clustering**

Each **mention** is:

$$
m = (n, u, A)
$$

Where:

* $n$: noun phrase (e.g., "President Obama")
* $u$: web URL
* $A$: attributes (i.e., $(\text{predicate}, \text{object})$ pairs)

#### Similarity Functions:

All operate on mention pairs $m, m'$.

* **String Identity**:

  $$
  f_{\text{strid}}(m, m') = 
  \begin{cases}
  1 & \text{if } n = n' \\
  0 & \text{otherwise}
  \end{cases}
  $$

* **Jaro-Winkler Similarity**: For fuzzy string matching

  $$
  f_{\text{strsim}}(m, m') = \text{JaroWinkler}(n, n')
  $$

* **Attribute Overlap** (Jaccard over (predicate, object)):

  $$
  f_{\text{attr}}(m, m') = \frac{|A \cap A'|}{|A \cup A'|}
  $$

* **IDF Token Overlap** (weighted word overlap):

  $$
  f_{\text{itok}}(m, m') = \frac{\sum_{w \in w(n) \cap w(n')} \text{IDF}(w)}{\sum_{w \in w(n) \cup w(n')} \text{IDF}(w)}
  $$

* **Word Overlap (page text)**: Jaccard over TF-IDF top-100 page words

* **Entity Overlap**: Jaccard over linked Freebase entities per page

* **Type Overlap**: Jaccard over inferred entity types



#### Combined Similarity:

$$
f_{\text{sim}}(m, m') = \sigma\left(c_0 + \sum_i c_i f_i(m, m')\right)
$$

Where:

* $\sigma$: Logistic function
* $f_i$: each feature described above
* $c_i$: learned coefficients via logistic regression


### Clustering Algorithm:

**Hierarchical Agglomerative Clustering (HAC)** using **single linkage**

* Merge clusters that have **maximum similarity**
* Use **token-based blocking (canopies)** to reduce pairwise comparisons


### Canonicalization:

Once a cluster is formed, choose the **canonical name** as:

* The noun phrase that appears in the **most distinct sources**
* If tie: choose the **longest phrase**


### B. **Relation Canonicalization via Rule Mining (AMIE)**

#### Step 1: Semi-canonicalize the KB

Subjects and objects are canonicalized, but predicates are still raw phrases.

#### Step 2: Use AMIE to find **subsumption rules**:

$$
r(x, y) \Rightarrow r'(x, y)
$$

If both directions exist: infer **equivalence**:

$$
r(x, y) \Leftrightarrow r'(x, y)
$$

Rules scored by **PCA Confidence**:

$$
\text{conf}_{\text{PCA}} = \frac{\text{support of rule}}{\text{support under PCA assumption}}
$$

This helps deal with KB **incompleteness**.

#### Step 3: Cluster relation phrases using transitive closure

#### Step 4: Map relation clusters to **Freebase schema** using:

* Matching patterns
* Co-occurrence
* Rule mining

## Comparison to Prior Work

| Method           | Entity Canonicalization | Relation Canonicalization | Notes                                         |
| ---------------- | ----------------------- | ------------------------- | --------------------------------------------- |
| Resolver         | Yes                     | Yes                       | Uses probabilistic modeling                   |
| Concept Resolver | Yes                     | No                        | Relies on NELL ontology                       |
| WEBRE            | Yes                     | Yes                       | Uses typed verb patterns                      |
| **This Paper**   | Yes                     | Yes                       | Combines OpenIE with rule mining and blocking |


## Experimental Setup

* **Corpus**: ReVerb triples over ClueWeb09 (millions of triples)
* **Gold standard**: Freebase-linked mentions and human evaluation
* **Two datasets**:

  * **Base**: clean entity data
  * **Ambiguous**: includes homonyms and polysemous names

### Evaluation Metrics:

Three levels for precision, recall, and F1:

* **Macro**: How many full clusters are pure
* **Micro**: Based on dominant entity per cluster
* **Pairwise**: Based on mention-pairs' correctness


## Results and Analysis

### Entity Clustering (on ReVerb):

| Feature           | Macro F1 | Micro F1 | Pairwise F1 |
| ----------------- | -------- | -------- | ----------- |
| String identity   | 0.61     | 0.89     | 0.85        |
| IDF token overlap | 0.93     | 0.98     | 0.99        |
| Attribute overlap | 0.10     | 0.37     | 0.17        |
| Type overlap      | 0.96     | 0.98     | 0.99        |
| **Simple ML**     | **0.94** | **0.98** | **0.99**    |

Takeaway: **IDF token overlap and type overlap are strongest individual features**


### Relation Clustering (on ReVerb):

| KB Type           | Conf. | Phrases | Clusters | Macro P | Micro P | Pairwise P |
| ----------------- | ----- | ------- | -------- | ------- | ------- | ---------- |
| Linked KB         | 0.8   | 522     | 118      | 0.90    | 0.94    | 0.95       |
| Linked KB + types | 0.8   | 752     | 303      | 0.95    | 0.98    | 0.997      |

Takeaway: **Adding type constraints improves relation clustering precision**


## Key Contributions

* A **practical, scalable method** for entity and relation canonicalization in OpenIE KBs
* Demonstrated that **simple ML features** + **blocking + rule mining** yield high-precision clusters
* Bridged **open** and **closed** IE systems with a hybrid model


## Limitations

* **Sparse attribute data** limits attribute overlap features
* **Ambiguous names** can reduce clustering precision
* Rule mining only covers a **subset** of relation phrases


## Summary

This paper proposes an efficient pipeline for:

* Clustering entity mentions into canonical groups
* Clustering synonymous relation phrases via rule mining
* Mapping canonical forms to a schema (e.g., Freebase)

The system is **modular**, scalable, and produces **high-precision clusters**—a strong step toward structuring Open IE data with minimal supervision.


## Implementing From Scratch-

In [1]:
import numpy as np
import pandas as pd
import re
import string
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Set
import itertools
import math
import networkx as nx
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist, squareform
import nltk
from nltk.corpus import stopwords
from nltk.metrics import jaccard_distance
from nltk.metrics.distance import jaro_winkler_similarity

nltk.download('stopwords', quiet=True) # Download stopwords if not already present
stop_words = set(stopwords.words('english')) # Initialising stop words

#### Loading Data

In [2]:
"""
Here we are defining a function to load triples and URLs.
By -> Tuple we are indicating that the function will return a tuple containing two lists:
1. A list of triples, where each triple is a tuple of three strings (subject, predicate, object).
2. A list of URLs, where each URL is a string.
"""
def load_data() -> Tuple[List[Tuple[str, str, str]], List[str]]:
    triples = [
        ("Obama", "was born in", "Honolulu"),
        ("Barack Obama", "place of birth", "Honolulu"),
        ("President Obama", "birthplace", "Honolulu"),
        ("Obama", "lives in", "Washington D.C."),
        ("Barack H. Obama", "resides in", "Washington"),
        ("Michelle Obama", "born in", "Chicago"),
        ("Obama", "married to", "Michelle Obama"),
        ("Michelle", "wife of", "Obama"),
        ("Washington", "is capital of", "United States"),
        ("Washington D.C.", "capital city of", "U.S."),
        ("Bombay", "located in", "India"),
        ("Mumbai", "is in", "Republic of India")
    ]

    urls = [
        "site1.com", "site2.com", "site3.com",
        "site1.com", "site4.com", "site5.com",
        "site1.com", "site6.com", "site7.com",
        "site8.com", "site9.com", "site10.com"
    ]
    return triples, urls

#### Mention Representation 
In information extraction, a mention refers to a specific occurrence of an entity (noun phrase) in text, along with its context and attributes.<br>
For example:
"Barack Obama was born in Honolulu" contains:
- Mention: "Barack Obama"
- Attributes: [("was born in", "Honolulu")]

In [3]:
class Mention:
    # Represents a mention of a noun phrase in text, along with its source and attributes
    def __init__(self, noun_phrase: str, source: str, attributes: List[Tuple[str, str]]):
        self.noun_phrase = noun_phrase # The noun phrase being mentioned
        self.source = source # The source of the mention (e.g., URL)
        self.attributes = attributes # List of attributes associated with the mention, each as a tuple (attribute_name, attribute_value) for (preicate, object)
        self.types = set() # Set of types associated with the mention, e.g., "Person", "Location"
    
    def __repr__(self):  # dfining  how an object is represented as a string
        # Displaying the noun phrase, source, and number of attributes like nametag of object with @ as separato and how many attributes it has for easy identification
        return f"{self.noun_phrase}@{self.source} [Attributes: {len(self.attributes)}]"
    
    def __hash__(self):
        # Hash based on noun phrase and source for uniqueness
        return hash((self.noun_phrase, self.source))
    
    def __eq__(self, other):
        return (self.noun_phrase == other.noun_phrase and self.source == other.source) # equality check based on noun phrase and source


def create_mentions(triples: List[Tuple[str, str, str]], urls: List[str]) -> List[Mention]:
    mention_map = defaultdict(list) # Creating a mapping from (noun_phrase, source) to attributes
    for (s, p, o), url in zip(triples, urls): # Iterating through each triple and its corresponding URL
        mention_map[(s, url)].append((p, o)) # Adding the predicate-object pair as an attribute for the subject

    mentions = [] # Create a list to hold Mention objects
    for (noun_phrase, source), attributes in mention_map.items(): # Creating Mention objects for each unique noun phrase and source combination
        mentions.append(Mention(noun_phrase, source, attributes)) # Appending the mention object to the list for mentions
    return mentions

#### Processing Tokens
Tokenizing text into normalized non-stopword tokens.

In [4]:
def tokenize(text: str) -> Set[str]: 
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text) # Removing punctuation from the text by re.sub which replaces punctuation with space 
    tokens = [word.lower() for word in text.split()
              if word.lower() not in stop_words and len(word) > 1] # Tokenizing the text into words, converting to lowercase, filtering out stopwords and single-character words
    return set(tokens) # Returning a set of tokens to ensure uniqueness

####  Token Blocking with Canopy Expansion
The Token Blocking with Canopy Expansion step is a critical optimization in the knowledge base canonicalization pipeline. Its purpose is to reduce the number of unnecessary comparisons between mentions before clustering them, making the process much faster while maintaining accuracy.<br>
This method works by creating "canopies" –clusters of similar mentions – based on tokens and then only comparing mentions within the same canopy or those that have tokens in common.<br>

**Purpose:**:<br>
1. **Token Blocking:** Groups mentions into "canopies" (blocks) based on shared keywords.
2. **Canopy Expansion:** Expands the blocking criteria by also considering object tokens (not just subject tokens).


In [7]:
def get_mention_tokens(mention:Mention) -> Set[str]: 
    # Function to get tokens from a mention with (subject + objects)
    tokens = tokenize(mention.noun_phrase) # Tokenizing the noun phrase of the mention
    for _, obj in mention.attributes: # Iterating through the attributes of the mention
        tokens.update(tokenize(obj)) # Adding tokens from the object part of the attributes
    return tokens # Returning the set of tokens for the mention

def create_canopies(mentions: List[Mention]) -> Dict[str, List[Mention]]: # Function to create canopies based on tokens using both subject and object tokens
    canopy_map = defaultdict(list) # Creating a mapping from token to mentions
    for mention in mentions: # Iterating through each mention
        tokens = get_mention_tokens(mention) # Getting the tokens for the mention
        for token in tokens: # Iterating through each token
            canopy_map[token].append(mention) # Adding the mention to the list of mentions for that token
    return canopy_map # Returning the mapping of tokens to mentions

def merge_canopies(canopy_map: Dict[str, List[Mention]]) -> List[Set[Mention]]: #Merging overlapping canopies using union-find
    # Create graph of mention connections
    G = nx.Graph()
    
    # Add all mentions as nodes
    all_mentions = set()
    for canopy in canopy_map.values():
        all_mentions.update(canopy) # Collecting all unique mentions from the canopy map
    
    for mention in all_mentions:
        G.add_node(mention) # Adding each mention as a node in the graph
    
    # Add edges between mentions in the same canopy
    for canopy_mentions in canopy_map.values():
        for m1, m2 in itertools.combinations(canopy_mentions, 2):
            G.add_edge(m1, m2) # Adding an edge between every pair of mentions in the same canopy
    
    # Return connected components as blocks
    return list(nx.connected_components(G))

#### Building Similarity Functions

Similarity functions are the core mechanism that determines how mentions (entity references) are grouped into clusters. They quantify how "alike" two mentions are based on different features like:
- String similarity (e.g., "Barack Obama" vs. "B. Obama")
- Attribute overlap (shared facts like birth place)
- Contextual similarity (co-occurring words or types)

In [8]:
def compute_df(mentions: List[Mention]) -> Dict[str, int]:
    #Computing document frequency for all tokens in the dataset
    freq = Counter()
    for m in mentions:
        tokens = get_mention_tokens(m) # Getting tokens for each mention
        for token in tokens:
            freq[token] += 1 # Incrementing the frequency count for each token
    return freq

#### Jaccard Attributes
Attribute Jaccard Similarity is a way to measure how similar two mentions are based on the facts (attributes) associated with them in a knowledge base. It is an adaptation of the Jaccard similarity which measures overlap between sets.<br>
In this paper, attributes are the (predicate, object) pairs that describe a noun phrase (mention) in OpenIE triples.

##### **Jaccard Similarity (The Core Idea)**

Given two sets $A$ and $B$, the **Jaccard similarity** is defined as:

$$
\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

* Numerator: number of **shared elements**
* Denominator: total number of **unique elements**

The value is between:

* $0$ (no overlap)
* $1$ (identical sets)

##### **What Are “Attributes” in This Context?**

In the paper which we are implementing a **mention** $m$ is a triple:

$$
m = (n, u, A)
$$

* $n$: noun phrase (e.g., “Barack Obama”)
* $u$: source URL where this mention appears
* $A$: set of attributes from that URL about $n$

Each attribute is a pair:

$$
(p, o) \in A \quad \text{where } p = \text{predicate, } o = \text{object}
$$

**Example**:

If a web page says:

> "Barack Obama was born in Honolulu and is married to Michelle Obama."

Then, the extracted attributes for mention “Barack Obama” could be:

$$
A = \{(\text{was born in}, \text{Honolulu}), (\text{is married to}, \text{Michelle Obama})\}
$$

##### **Attribute Jaccard Similarity: The Formula**

Given two mentions:

$$
m = (n, u, A), \quad m' = (n', u', A')
$$

Then the **attribute Jaccard similarity** is:

$$
f_{\text{attr}}(m, m') = \frac{|A \cap A'|}{|A \cup A'|}
$$

Where:

* $A$ and $A'$ are sets of (predicate, object) pairs
* Matching requires both predicate and object to be **exactly equal**

##### ** Intuition Through Example**

Let’s assume that:

**Mention 1 (m):**

* $n = \text{Barack Obama}$
* $A = \{(\text{born in}, \text{Hawaii}), (\text{president of}, \text{USA})\}$

**Mention 2 (m′):**

* $n′ = \text{President Obama}$
* $A' = \{(\text{born in}, \text{Hawaii}), (\text{lives in}, \text{Washington D.C.})\}$

Then:

$$
A \cap A' = \{(\text{born in}, \text{Hawaii})\} \\
A \cup A' = \{(\text{born in}, \text{Hawaii}), (\text{president of}, \text{USA}), (\text{lives in}, \text{Washington D.C.})\}
$$

So:

$$
f_{\text{attr}}(m, m') = \frac{1}{3} \approx 0.33
$$

This indicates **partial similarity** — they share one key attribute.


In [9]:
def jaccard_attributes(m1: Mention, m2: Mention) -> float:
    # calculating jaccard similarity
    a1 = set((p, o) for p, o in m1.attributes) # Creating a set of (predicate, object) pairs for the first mention
    a2 = set((p, o) for p, o in m2.attributes)
    if not a1 or not a2:
        return 0.0
    intersection = len(a1 & a2) # Calculating the intersection of attributes
    union = len(a1 | a2) # Calculating the union of attributes
    return intersection / union

#### Jaro Winkler String Similarity
It’s a **string similarity metric** designed to compare two short strings (e.g., names) and return a value between **0** and **1**:

* **1** means identical strings
* **0** means completely different

It’s especially good for **matching person names**, **spelling variations**, or **typos**—and that’s why the canonicalization paper uses it to compare noun phrases like:

* “Barack Obama” vs. “B. Obama”
* “Jonathon” vs. “Jonathan”

##### *Why Don't We Just Use Edit Distance?*
**Edit distance** (Levenshtein) counts how many character insertions/deletions/substitutions are needed to turn one string into another but it’s **computationally expensive** and doesn’t favor **prefix matches**, which are important in names.

**Jaro-Winkler** on other hand is **faster** than edit distance and it gives **higher scores** when the beginning of the strings match (prefix boosting).




##### **Jaro Similarity**

Given two strings $s$ and $t$, the **Jaro similarity** is calculated based on:

* $m$: number of **matching characters**
* $t$: number of **transpositions** (mismatched order)


1. Step 1: Matching Window

Characters are considered "matching" if they are the same and not too far apart.
Defining a **window size**:

$$
\text{window} = \left\lfloor \frac{\max(|s|, |t|)}{2} \right\rfloor - 1
$$

Within this window, characters are eligible to match.

2. Match Count $m$

Count how many characters in $s$ and $t$ are **equal** and **within the window**.

3. Transpositions $t$

Count how many matching characters are **out of order**, and divide by 2.

4. Jaro Score

$$
\text{Jaro}(s, t) = \frac{1}{3} \left( \frac{m}{|s|} + \frac{m}{|t|} + \frac{m - t}{m} \right)
$$

If no matches: Jaro = 0.

##### **Jaro-Winkler Similarity**

Jaro-Winkler improves on Jaro by adding a **prefix bonus**.

Let $l$ be the length of the **common prefix** (up to 4 characters), and $p$ a **scaling factor** (usually $p = 0.1$).

$$
\text{Jaro-Winkler}(s, t) = \text{Jaro}(s, t) + l \cdot p \cdot (1 - \text{Jaro}(s, t))
$$

This boosts similarity when the strings **start the same** — common for person or entity names.

##### **Example**

Compare:

* $s = \text{"MARTHA"}$
* $t = \text{"MARHTA"}$

##### Step-by-step:

* Matching characters (window = 2):
  M, A, R, T, H, A → **all match** → $m = 6$

* Transpositions:
  H and T are swapped → 2 mismatches → $t = 1$

So:

$$
\text{Jaro}(s, t) = \frac{1}{3} \left( \frac{6}{6} + \frac{6}{6} + \frac{6 - 1}{6} \right)
= \frac{1}{3}(1 + 1 + 0.8333) = 0.9444
$$

Now compute common prefix $l = 3$ (MAR) and $p = 0.1$:

$$
\text{Jaro-Winkler} = 0.9444 + 3 \cdot 0.1 \cdot (1 - 0.9444) = 0.9611
$$

So the final similarity is **0.9611** — very close strings.

**Why Useful in the Canonicalization Paper**

In the paper we are implementing noun phrases like:

* "President Obama" vs "Barack H. Obama"
* "USA" vs "United States"

don’t match exactly, but have overlapping substrings. Here Jaro-Winkler helps in:
* Handle **spelling variation**
* Boost matches with **shared prefixes**
* Avoid overly strict string identity

It becomes one of the **features** used in logistic regression to predict if two mentions should be clustered.


In [10]:
def string_sim(m1: Mention, m2: Mention) -> float:
    #tring similarity using Jaro-Winkler
    return jaro_winkler_similarity(m1.noun_phrase, m2.noun_phrase)